# A landslide in Ancona, seen point by point

In 1982 a large landslide moved a whole hillside of Ancona (the *Posatora* landslide), and
the slope is still monitored. Here we use **L2B**: individual measurement points (buildings,
rocks, poles) rather than a 100 m grid, each with its own time series.

L2 points are measured **along the satellite's line of sight**. Each place is seen from two
geometries, the satellite going north (*ascending*) and going south (*descending*), so we take
one burst of each.

**Downloads:** ~170 MB (two bursts), nothing afterwards.

*Data: EGMS, © European Union, Copernicus Land Monitoring Service. Anything you publish from it must credit the source.*

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer

from egms_client import EgmsArchive, Query

# The key comes from EGMS_SERVICE_KEY or EGMS_SERVICE_KEY_FILE (see the README).
# Downloads are kept here: run the notebook again and nothing is downloaded.
DATA = Path(os.environ.get("EGMS_DATA_DIR", "egms-data"))
RELEASE = "2020-2025"
to_laea = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)

In [ ]:
SLOPE = (13.478, 43.607)  # lon, lat: the Posatora hillside
BOX = (13.45, 43.59, 13.50, 43.62)
query = Query(levels=("L2B",), release=RELEASE, bbox=BOX)

async with EgmsArchive() as archive:
    hits = [h for s in await archive.find(query) for h in s.hits]

# The smallest burst of each direction whose footprint covers the slope. A burst is a
# tilted parallelogram: hit.covers() checks its real outline, not just its bbox.
chosen = {}
for hit in sorted(hits, key=lambda h: h.filesize):
    if hit.filesize > 1_000_000 and hit.covers(*SLOPE):
        chosen.setdefault(hit.direction, hit)
for direction, hit in chosen.items():
    print(f"{direction:>10}: {hit.filename} ({hit.filesize / 1e6:.0f} MB)")

In [ ]:
import pandas as pd

async with EgmsArchive() as archive:
    paths = {}
    for direction, hit in chosen.items():
        burst = Query(
            levels=("L2B",),
            release=RELEASE,
            relative_orbit=hit.relative_orbit,
            burst_cycle=hit.burst_cycle,
            swath=hit.swath,
        )
        (paths[direction],) = await archive.fetch(burst, DATA, keep=["parquet"])

x0, y0 = to_laea.transform(*SLOPE)
points = {}
for direction, path in paths.items():
    p = pd.read_parquet(path).drop(columns="geometry")
    near = p[(np.hypot(p.easting - x0, p.northing - y0) < 1500) & (p.temporal_coherence > 0.7)]
    points[direction] = near
    print(f"{direction:>10}: {len(near):,} reliable points within 1.5 km")

## Where the ground moves

Colour is the mean speed along the line of sight, in mm/year: **negative means moving away
from the satellite** (on a slope, mostly downhill), positive towards it. Points that barely move
are pale.

In [ ]:
fig, axes = plt.subplots(1, len(points), figsize=(13, 5.5), sharex=True, sharey=True)
for ax, (direction, p) in zip(np.atleast_1d(axes), points.items(), strict=True):
    im = ax.scatter(p.easting, p.northing, c=p.mean_velocity, cmap="RdBu", vmin=-10, vmax=10, s=6)
    ax.plot(x0, y0, "k+", markersize=14)
    ax.set_title(f"{direction} (line of sight)")
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.colorbar(im, ax=axes, label="mean velocity along the line of sight (mm/year)")

## Moving points against stable ones

The 30 fastest points next to the 30 stillest ones, date by date.

In [ ]:
from egms_client.tables import to_xarray as series_to_xarray

p = points["descending"] if "descending" in points else next(iter(points.values()))
moving = series_to_xarray(p.nsmallest(30, "mean_velocity")).mean("pid")
still = series_to_xarray(p.loc[p.mean_velocity.abs().nsmallest(30).index]).mean("pid")

fig, ax = plt.subplots(figsize=(11, 4))
moving.plot(ax=ax, label="30 fastest points", color="#2a78d6")
still.plot(ax=ax, label="30 stillest points", color="#888780")
ax.set_ylabel("displacement along the line of sight (mm)")
ax.set_xlabel("")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title(
    f"Within 1.5 km of the Posatora slope: {float(moving[-1] - moving[0]):.0f} mm in the fastest points"
)
fig.text(
    0.01,
    -0.02,
    "EGMS L2B 2020-2025. Contains modified Copernicus Land Monitoring Service information.",
    fontsize=8,
)
fig.savefig("../docs/images/notebook-landslide-ancona.png", dpi=100, bbox_inches="tight")

Things to try:
- change `SLOPE` and `BOX` to a landslide you know;
- compare the ascending and descending speeds: combined, they give the vertical and
  east-west motion (that is what L3 does, on a 100 m grid);
- filter on `temporal_coherence` more or less strictly and see what changes.